# MicroReasoner Smoke Notebook

Fast preflight before long runs: tiny data -> build -> SFT -> GRPO -> resume -> validate -> final eval -> demo.

Commands covered: `data build-sft`, `data build-rl`, `train sft`, `train grpo`, `validate-run`, `scripts/run_final_evaluation.py`, `scripts/run_demo_compare.py`.

Optional real-backend 1-step SFT/GRPO is included (off by default).

In [ ]:
import json, subprocess, sys, time
from pathlib import Path

if not Path("pyproject.toml").exists() or not Path("src/microreasoner").exists():
    raise RuntimeError("Run from MicroReasoner repo root")

RUN_REAL_BACKEND_SMOKE = False
ALLOW_NETWORK_DOWNLOADS = False
REAL_BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[dev]"])

def run_cmd(cmd):
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(cmd)}")

def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

STAMP = str(int(time.time()))
RID = {
    "bs": f"smoke-{STAMP}-build-sft",
    "br": f"smoke-{STAMP}-build-rl",
    "s": f"smoke-{STAMP}-sft",
    "g": f"smoke-{STAMP}-grpo",
    "gr": f"smoke-{STAMP}-grpo-resume",
    "f": f"smoke-{STAMP}-final",
    "d": f"smoke-{STAMP}-demo",
    "rs": f"smoke-{STAMP}-real-sft",
    "rg": f"smoke-{STAMP}-real-grpo",
}
ROOT = Path(".")
RAW = ROOT / "artifacts" / "smoke" / "raw"
EVAL = ROOT / "artifacts" / "smoke" / "eval"
REP = ROOT / "reports" / "smoke"
for d in (RAW, EVAL, REP):
    d.mkdir(parents=True, exist_ok=True)
print("Smoke IDs:", RID)

In [ ]:
def write_jsonl(path: Path, rows):
    with path.open("w", encoding="utf-8") as h:
        for r in rows:
            h.write(json.dumps(r, ensure_ascii=False) + "\n")

def canon(prefix, bench, n, start):
    out = []
    for i in range(n):
        a, b = start + i + 2, (i * 3) % 17 + 1
        q = f"Compute {a} * {b}." if bench == "math" else f"Compute {a} + {b}."
        ans = str(a * b) if bench == "math" else str(a + b)
        out.append({"id": f"{prefix}_{i:04d}", "question": q, "think": "reason", "answer_boxed": ans, "benchmark": bench, "source_name": prefix, "metadata": {"difficulty": "easy"}})
    return out

write_jsonl(RAW / "openr1_math_subset.jsonl", canon("openr1_math_subset", "math", 50, 0))
write_jsonl(RAW / "gsm8k_small_mix.jsonl", canon("gsm8k_small_mix", "gsm8k", 50, 100))
write_jsonl(RAW / "math_small_mix.jsonl", canon("math_small_mix", "math", 50, 200))

gsm = []
for i in range(8):
    x, y, ans = i + 3, i + 5, str(i + 8)
    gsm.append({"id": f"gsm_eval_{i:03d}", "question": f"Compute {x} + {y}.", "answer": ans, "mock_greedy_response": f"<think>x</think>\n<answer>\\boxed{{{ans}}}</answer>", "mock_sampled_responses": [f"<think>x</think>\n<answer>\\boxed{{{ans}}}</answer>", "<think>x</think>\n<answer>\\boxed{0}</answer>"]})
mat = []
for i in range(8):
    x, y, ans = i + 4, i + 6, str((i + 4) * (i + 6))
    mat.append({"id": f"math_eval_{i:03d}", "question": f"Compute {x} * {y}.", "answer": ans, "mock_greedy_response": f"<think>x</think>\n<answer>\\boxed{{{ans}}}</answer>", "mock_sampled_responses": [f"<think>x</think>\n<answer>\\boxed{{{ans}}}</answer>", "<think>x</think>\n<answer>\\boxed{1}</answer>"]})
write_jsonl(EVAL / "gsm8k_eval.jsonl", gsm)
write_jsonl(EVAL / "math_eval.jsonl", mat)
print("Raw/eval smoke data written")

In [ ]:
run_cmd([sys.executable, "-m", "microreasoner.cli.main", "data", "build-sft", "--config", "configs/defaults.yaml", "--source-dir", str(RAW), "--run-id", RID["bs"]])
run_cmd([sys.executable, "-m", "microreasoner.cli.main", "data", "build-rl", "--config", "configs/defaults.yaml", "--source-dir", str(RAW), "--run-id", RID["br"]])
sft_build = Path("artifacts/runs") / RID["bs"]
rl_build = Path("artifacts/runs") / RID["br"]
SFT_MAN = Path(read_json(sft_build / "summary.json")["artifacts"]["dataset_manifest_path"])
RL_MAN = Path(read_json(rl_build / "summary.json")["artifacts"]["dataset_manifest_path"])
run_cmd([sys.executable, "-m", "microreasoner.cli.main", "data", "inspect", "--dataset-manifest", str(SFT_MAN)])
run_cmd([sys.executable, "-m", "microreasoner.cli.main", "data", "inspect", "--dataset-manifest", str(RL_MAN)])

run_cmd([sys.executable, "-m", "microreasoner.cli.main", "train", "sft", "--config", "configs/defaults.yaml", "--dataset-manifest", str(SFT_MAN), "--run-id", RID["s"], "--max-steps", "8", "--eval-every-steps", "2", "--set", "train_sft.backend.trainer=fixture", "--set", "train_sft.gates.schema_min=0.0", "--set", "train_sft.run.max_eval_samples=16"])
sft_run = Path("artifacts/runs") / RID["s"]
SFT_BEST = Path(read_json(sft_run / "checkpoints.json")["best"])
run_cmd([sys.executable, "-m", "microreasoner.cli.main", "validate-run", "--run-dir", str(sft_run)])

run_cmd([sys.executable, "-m", "microreasoner.cli.main", "train", "grpo", "--config", "configs/defaults.yaml", "--dataset-manifest", str(RL_MAN), "--init-checkpoint", str(SFT_BEST), "--run-id", RID["g"], "--max-steps", "8", "--eval-every-steps", "2", "--set", "train_grpo.backend.trainer=fixture", "--set", "train_grpo.batch.per_device=1", "--set", "train_grpo.batch.grad_accum=1", "--set", "train_grpo.algo.group_size=4", "--set", "train_grpo.batch.max_prompt_len=128", "--set", "train_grpo.batch.max_completion_len=64", "--set", "train_grpo.gates.min_schema_compliance_rate=0.0", "--set", "train_grpo.gates.max_parser_failure_rate=1.0", "--set", "train_grpo.gates.min_reward_std=0.0", "--set", "train_grpo.run.max_eval_samples=16"])
grpo_run = Path("artifacts/runs") / RID["g"]
GRPO = read_json(grpo_run / "checkpoints.json")
GRPO_BEST = Path(GRPO["best"])
GRPO_LATEST = Path(GRPO["latest"])
run_cmd([sys.executable, "-m", "microreasoner.cli.main", "validate-run", "--run-dir", str(grpo_run)])

run_cmd([sys.executable, "-m", "microreasoner.cli.main", "train", "grpo", "--config", "configs/defaults.yaml", "--dataset-manifest", str(RL_MAN), "--init-checkpoint", str(SFT_BEST), "--resume-from", str(GRPO_LATEST), "--run-id", RID["gr"], "--max-steps", "10", "--eval-every-steps", "2", "--set", "train_grpo.backend.trainer=fixture", "--set", "train_grpo.batch.per_device=1", "--set", "train_grpo.batch.grad_accum=1", "--set", "train_grpo.algo.group_size=4", "--set", "train_grpo.batch.max_prompt_len=128", "--set", "train_grpo.batch.max_completion_len=64", "--set", "train_grpo.gates.min_schema_compliance_rate=0.0", "--set", "train_grpo.gates.max_parser_failure_rate=1.0", "--set", "train_grpo.gates.min_reward_std=0.0", "--set", "train_grpo.run.max_eval_samples=16"])
grpo_resume = Path("artifacts/runs") / RID["gr"]
run_cmd([sys.executable, "-m", "microreasoner.cli.main", "validate-run", "--run-dir", str(grpo_resume)])

base_ckpt = Path("artifacts/smoke/checkpoints/base_fixture")
base_ckpt.mkdir(parents=True, exist_ok=True)
(base_ckpt / "README.txt").write_text("fixture\n", encoding="utf-8")

run_cmd([sys.executable, "scripts/run_final_evaluation.py", "--config", "configs/defaults.yaml", "--dataset-dir", str(EVAL), "--base-checkpoint", str(base_ckpt), "--sft-checkpoint", str(SFT_BEST), "--grpo-checkpoint", str(GRPO_BEST), "--output-root", "artifacts/smoke/final_eval", "--report-dir", str(REP), "--mode", "fixture", "--max-items", "8", "--session-id", RID["f"]])

demo_args = [
    "--config", "configs/defaults.yaml",
    "--dataset-dir", str(EVAL),
    "--base-checkpoint", str(base_ckpt),
    "--sft-checkpoint", str(SFT_BEST),
    "--grpo-checkpoint", str(GRPO_BEST),
    "--output-root", "artifacts/smoke/demo",
    "--output-file", str(REP / "demo_compare.md"),
    "--mode", "fixture",
    "--max-items", "8",
    "--session-id", RID["d"],
]
demo_ran = False
if Path("scripts/run_demo_compare.py").exists():
    run_cmd([sys.executable, "scripts/run_demo_compare.py", *demo_args])
    demo_ran = True
elif Path("src/microreasoner/demo/app.py").exists():
    run_cmd([sys.executable, "-m", "microreasoner.demo.app", *demo_args])
    demo_ran = True
else:
    print("Skipping demo compare smoke: script/module not present in this snapshot")

required_reports = [REP / "final_metrics.json", REP / "final_report.md", REP / "error_analysis.md"]
if demo_ran:
    required_reports.append(REP / "demo_compare.md")
for p in required_reports:
    if not p.exists():
        raise RuntimeError(f"Missing expected report: {p}")
final_metrics = read_json(REP / "final_metrics.json")
if final_metrics.get("status") != "success":
    raise RuntimeError(f"Final evaluation reported status={final_metrics.get('status')}: {final_metrics.get('failure_reasons')}")
print("Fixture smoke passed")

In [ ]:
if not RUN_REAL_BACKEND_SMOKE:
    print("Skipping optional real-backend smoke")
else:
    if not ALLOW_NETWORK_DOWNLOADS:
        raise RuntimeError("Set ALLOW_NETWORK_DOWNLOADS=True before real-backend smoke")
    req = [
        "transformers==4.46.0", "datasets==2.21.0", "accelerate==0.34.0", "peft==0.11.0",
        "trl==0.14.0", "bitsandbytes==0.48.1", "sentencepiece==0.2.0", "protobuf==4.25.0",
        "huggingface_hub==0.24.0", "math-verify==0.6.0"
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *req])
    run_cmd([sys.executable, "-m", "microreasoner.cli.main", "train", "sft", "--config", "configs/defaults.yaml", "--dataset-manifest", str(SFT_MAN), "--run-id", RID["rs"], "--max-steps", "1", "--eval-every-steps", "1", "--set", f"model.default_base_model={REAL_BASE_MODEL}", "--set", "train_sft.backend.trainer=transformers", "--set", "train_sft.mode=qlora", "--set", "train_sft.batch.per_device=1", "--set", "train_sft.batch.grad_accum=1", "--set", "train_sft.batch.max_seq_len=256", "--set", "train_sft.gates.schema_min=0.0", "--set", "evaluation.sampled.num_samples=2"])
    real_sft = Path(read_json((Path("artifacts/runs") / RID["rs"] / "checkpoints.json"))["best"])
    run_cmd([sys.executable, "-m", "microreasoner.cli.main", "train", "grpo", "--config", "configs/defaults.yaml", "--dataset-manifest", str(RL_MAN), "--init-checkpoint", str(real_sft), "--run-id", RID["rg"], "--max-steps", "1", "--eval-every-steps", "1", "--set", f"model.default_base_model={REAL_BASE_MODEL}", "--set", "train_grpo.backend.trainer=trl", "--set", "train_grpo.batch.per_device=1", "--set", "train_grpo.batch.grad_accum=1", "--set", "train_grpo.algo.group_size=2", "--set", "train_grpo.batch.max_prompt_len=128", "--set", "train_grpo.batch.max_completion_len=64", "--set", "train_grpo.gates.min_schema_compliance_rate=0.0", "--set", "train_grpo.gates.max_parser_failure_rate=1.0", "--set", "train_grpo.gates.min_reward_std=0.0", "--set", "evaluation.sampled.num_samples=2"])
    print("Real-backend micro smoke passed")

If this notebook passes, your wiring/checkpoint/resume/eval path is healthy. Then run the full notebook.